<a href="https://colab.research.google.com/github/ianleonardo/5m-data-1.1-intro-data-science/blob/main/ADK_Learning_tools.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 Welcome to Your ADK Adventure - Tools & Memory! 🚀

Welcome, Agent Architect! This notebook is your guide to giving your AI agents two essential superpowers: custom tools and conversational memory.

By the end of this adventure, you will be able to:

- **Build a Foundational Agent**: Create a simple but effective AI agent from scratch using the Google Agent Development Kit (ADK).

- **Grant New Skills with Custom Tools**: Teach an agent to perform new tasks by connecting it to external APIs, like a real-time weather service.

- **Create a Team of Agents**: Assemble a multi-agent system where a primary agent can delegate specialized tasks to other agents.

- **Master Conversational Memory**: Understand the critical role of Sessions in enabling agents to remember previous interactions, handle feedback, and carry on a coherent conversation.


Let's get this adventure started!

## Author

HI, I'm Qingyue (Annie) Wang, a developer advocate and AI engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)


If you have questions with this notebook, contact me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/) , [X](https://twitter.com/anniewangtech) or email anniewangtech0510@Gmail.com


```
  (\__/)
  (•ㅅ•)
  /づ  📚      Enjoy learning AI Agents :)
```


-------------
### 🎁 🛑 Important Prerequisite: Setup Your Environment! 🛑 🎁
-----------------------------------------------------------------------------

👉 **Get Your API Key HERE**: https://codelabs.developers.google.com/onramp/instructions#1

 -----------------------------------------------------------------------------

```
 ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️  ⬆️
   /\_/\     /\_/\     /\_/\      /\_/\       /\_/\
  ( ^_^ )   ( -.- )   ( >_< )   ( =^.^= )    ( o_o )             
```


## Part 0: Setup & Authentication 🔑

First things first, let's get all our tools ready. This step installs the necessary libraries and securely configures your Google API key so your agents can access the power of Gemini.

In [3]:
!pip install google-adk google-generativeai -q

# --- Import all necessary libraries ---
import os
import sys
import json
import asyncio
import random
import string
from uuid import uuid4
from typing import Any, List

import pandas as pd
import plotly.graph_objects as go
import vertexai
from google.colab import auth
from IPython.display import HTML, Markdown, display

# --- ADK, Agent, and Evaluation Components ---
from google.adk.agents import Agent
from google.adk.events import Event
from google.adk.runners import Runner
import google.adk as adk
from google.adk.tools import google_search
from google.adk.sessions import InMemorySessionService, Session
from google.genai import types
from google.genai.types import Content, Part


print("✅ All libraries are ready to go!")


✅ All libraries are ready to go!


### Authenticate and Configure Your Project
To use Vertex AI, you need an active Google Cloud project. This section handles authenticating your environment and setting up the necessary project configurations.

In [4]:
# ---  Authentication & Project Configuration ---

# Authenticate user in Colab
if "google.colab" in sys.modules:
    auth.authenticate_user()
    print("✅ Authenticated successfully.")

✅ Authenticated successfully.


In [10]:
# @title Set Your Google Cloud Project Details
PROJECT_ID = "gen-lang-client-0186681808"             # @param {type:"string"}
LOCATION = "asia-southeast1"               # @param {type:"string"}

# Set environment variables for the ADK and gcloud
os.environ["GOOGLE_CLOUD_PROJECT"] = PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = LOCATION
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "True"

!gcloud services enable aiplatform.googleapis.com --project={PROJECT_ID}

print(f"\n✅ Vertex AI configured for project '{PROJECT_ID}' in '{LOCATION}'.")


✅ Vertex AI configured for project 'gen-lang-client-0186681808' in 'asia-southeast1'.


In [ ]:
!gcloud auth login

Go to the following link in your browser, and complete the sign-in prompts:

    https://accounts.google.com/o/oauth2/auth?response_type=code&client_id=32555940559.apps.googleusercontent.com&redirect_uri=https%3A%2F%2Fsdk.cloud.google.com%2Fauthcode.html&scope=openid+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fuserinfo.email+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcloud-platform+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fappengine.admin+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fsqlservice.login+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fcompute+https%3A%2F%2Fwww.googleapis.com%2Fauth%2Faccounts.reauth&state=W9Ul0mpERSTXMYtrs4J7Yi96SpkyFg&prompt=consent&token_usage=remote&access_type=offline&code_challenge=Wa7xV2MqcGKqrTdqGLU6m-CpzIQPtM0d7dAD_60YOAI&code_challenge_method=S256

Once finished, enter the verification code provided in your browser: 4/0AeoWuM91Dp0e0vPAPGf0CsmhSKX969H5RNcEWITw-vDHzfl_GfS7zHYbNl1jHF_-k8curQ

You are now logged in as [ianleonardo@gmail.com].
Your current project 

---
## Part 1: Your First Agent - The Day Trip Genie 🧞

Meet your first creation! The `day_trip_agent` is a simple but powerful assistant. We're making it a little smarter by teaching it to understand **budget constraints**.

* **Agent**: The brain of the operation, defined by its instructions, tools, and the AI model it uses.
* **Session**: The conversation history. For this simple agent, it's just a container for a single request-response.
* **Runner**: The engine that connects the `Agent` and the `Session` to process your request and get a response.

```
+--------------------------------------------------+
|         Spontaneous Day Trip Agent 🤖            |
|--------------------------------------------------|
|  Model: gemini-2.5-flash                         |
|  Description:                                    |
|   Generates full-day trip itineraries based on   |
|   mood, interests, and budget                    |
|--------------------------------------------------|
|  🔧 Tools:                                       |
|   - Google Search                                |
|--------------------------------------------------|
|  🧠 Capabilities:                                |
|   - Budget Awareness (cheap / splurge)           |
|   - Mood Matching (adventurous, relaxing, etc.)  |
|   - Real-Time Info (hours, events)               |
|   - Morning / Afternoon / Evening plan           |
+--------------------------------------------------+

            ▲
            |
    +------------------+
    |   User Input     |
    |------------------|
    |  Mood            |
    |  Interests       |
    |  Budget          |
    +------------------+

            |
            ▼

+--------------------------------------------------+
|             Output: Markdown Itinerary           |
|--------------------------------------------------|
| - Time blocks (Morning / Afternoon / Evening)    |
| - Venue names with links and hours               |
| - Budget-matching activities                     |
+--------------------------------------------------+
```


In [7]:
# --- Agent Definition ---

def create_day_trip_agent():
    """Create the Spontaneous Day Trip Generator agent"""
    return Agent(
        name="day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent specialized in generating spontaneous full-day itineraries based on mood, interests, and budget.",
        instruction="""
        You are the "Spontaneous Day Trip" Generator 🚗 - a specialized AI assistant that creates engaging full-day itineraries.

        Your Mission:
        Transform a simple mood or interest into a complete day-trip adventure with real-time details, while respecting a budget.

        Guidelines:
        1. **Budget-Aware**: Pay close attention to budget hints like 'cheap', 'affordable', or 'splurge'. Use Google Search to find activities (free museums, parks, paid attractions) that match the user's budget.
        2. **Full-Day Structure**: Create morning, afternoon, and evening activities.
        3. **Real-Time Focus**: Search for current operating hours and special events.
        4. **Mood Matching**: Align suggestions with the requested mood (adventurous, relaxing, artsy, etc.).

        RETURN itinerary in MARKDOWN FORMAT with clear time blocks and specific venue names.
        """,
        tools=[google_search]
    )

day_trip_agent = create_day_trip_agent()
print(f"🧞 Agent '{day_trip_agent.name}' is created and ready for adventure!")

🧞 Agent 'day_trip_agent' is created and ready for adventure!


In [11]:
# --- A Helper Function to Run Our Agents ---
# We'll use this function throughout the notebook to make running queries easy.

async def run_agent_query(agent: Agent, query: str, session: Session, user_id: str, is_router: bool = False):
    """Initializes a runner and executes a query for a given agent and session."""
    print(f"\n🚀 Running query for agent: '{agent.name}' in session: '{session.id}'...")

    runner = Runner(
        agent=agent,
        session_service=session_service,
        app_name=agent.name
    )

    final_response = ""
    try:
        async for event in runner.run_async(
            user_id=user_id,
            session_id=session.id,
            new_message=Content(parts=[Part(text=query)], role="user")
        ):
            if not is_router:
                # Let's see what the agent is thinking!
                print(f"EVENT: {event}")
            if event.is_final_response():
                final_response = event.content.parts[0].text
    except Exception as e:
        final_response = f"An error occurred: {e}"

    if not is_router:
     print("\n" + "-"*50)
     print("✅ Final Response:")
     display(Markdown(final_response))
     print("-"*50 + "\n")

    return final_response

# --- Initialize our Session Service ---
# This one service will manage all the different sessions in our notebook.
session_service = InMemorySessionService()
my_user_id = "adk_adventurer_001"

In [12]:
# --- Let's test the Day Trip Genie! ---

async def run_day_trip_genie():
    # Create a new, single-use session for this query
    day_trip_session = await session_service.create_session(
        app_name=day_trip_agent.name,
        user_id=my_user_id
    )

    # Note the new budget constraint in the query!
    query = "Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!"
    print(f"🗣️ User Query: '{query}'")

    await run_agent_query(day_trip_agent, query, day_trip_session, my_user_id)

await run_day_trip_genie()

🗣️ User Query: 'Plan a relaxing and artsy day trip near Sunnyvale, CA. Keep it affordable!'

🚀 Running query for agent: 'day_trip_agent' in session: '78968f43-7d7b-49e9-b8ad-397bec80a1c8'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Here's a relaxing and artsy, yet affordable, day trip itinerary near Sunnyvale, CA, designed for today, Monday, May 18, 2026:

## Relaxing & Artsy Day Trip near Sunnyvale, CA

This itinerary balances artistic exploration with peaceful moments, all while keeping your budget in mind.

### **Morning (9:00 AM - 12:00 PM): Local Art & Sunnyvale's Sculptures**

Start your day with a local, affordable art immersion right in Sunnyvale.

*   **9:00 AM - 10:00 AM: Coffee and Local Art at Sunnyvale Art Gallery**
    Begin your day at the Sunnyvale Art Gallery. Described as an art gallery, cafe, art supply store, and flower shop, it offers a spacious and quiet environment. You can grab an affordable coffee or beverage an

Here's a relaxing and artsy, yet affordable, day trip itinerary near Sunnyvale, CA, designed for today, Monday, May 18, 2026:

## Relaxing & Artsy Day Trip near Sunnyvale, CA

This itinerary balances artistic exploration with peaceful moments, all while keeping your budget in mind.

### **Morning (9:00 AM - 12:00 PM): Local Art & Sunnyvale's Sculptures**

Start your day with a local, affordable art immersion right in Sunnyvale.

*   **9:00 AM - 10:00 AM: Coffee and Local Art at Sunnyvale Art Gallery**
    Begin your day at the Sunnyvale Art Gallery. Described as an art gallery, cafe, art supply store, and flower shop, it offers a spacious and quiet environment. You can grab an affordable coffee or beverage and browse local artists' works in their rotating exhibitions.
    *   **Location:** Sunnyvale Art Gallery, 251 W El Camino Real, Sunnyvale, CA.
    *   **Hours:** Monday - Friday, 8 AM - 7 PM.
    *   **Cost:** Cost of coffee/small purchase.
*   **10:00 AM - 12:00 PM: Self-Guided Public Art Tour in Sunnyvale**
    After your coffee, embark on a relaxing walk to discover Sunnyvale's "Sun Flair" public art program. This program features 26 identical sun sculptures, each uniquely transformed by local artists, installed in city parks until early 2027. You can find walking tour maps online that provide detailed instructions for exploring pieces in areas like Civic Center and Washington Park, or Downtown and Murphy Park. This offers a free and gentle stroll through local green spaces while engaging with diverse artistic expressions.
    *   **Location:** Various parks in Sunnyvale (e.g., Civic Center, Washington Park, Downtown, Murphy Park). Refer to Sunnyvale's Public Art webpage for maps.
    *   **Cost:** Free.

### **Lunch (12:00 PM - 1:00 PM): Affordable Fuel-Up**

For an affordable and relaxing lunch, consider bringing your own picnic to enjoy in one of Sunnyvale's parks, or grab a quick, inexpensive bite on your way to the next destination.

*   **Suggestion:** Pick up sandwiches or snacks from a local grocery store or a fast-casual eatery.

### **Afternoon (1:00 PM - 4:30 PM): World-Class Art at Stanford**

Head over to Stanford University for a truly world-class art experience that is completely free.

*   **1:00 PM - 4:30 PM: Cantor Arts Center & Rodin Sculpture Garden**
    The Cantor Arts Center at Stanford University is a premier arts destination with free admission. While some older information indicated it might be closed on Mondays, recent visitor updates confirm the Cantor will be open on Mondays (including federal holidays) from 11 AM - 6 PM. Explore its extensive collection, which spans 5,000 years of art, and don't miss the renowned outdoor Rodin Sculpture Garden, featuring over 20 bronze sculptures by Auguste Rodin, for a serene and contemplative artistic experience.
    *   **Location:** 328 Lomita Dr, Stanford, CA 94305.
    *   **Hours:** Mondays, 11 AM - 6 PM.
    *   **Cost:** Admission is free. Parking on weekdays (until 4 PM) uses the ParkMobile app and incurs a fee.

### **Late Afternoon / Evening (4:30 PM Onwards): Relaxing Stroll & Affordable Dinner**

Wind down your day with a final relaxing activity and an affordable dinner.

*   **4:30 PM - 6:00 PM: Stroll the Stanford Campus Grounds**
    After the museum, take a leisurely stroll through the beautiful Stanford University campus. The architecture, manicured gardens, and open spaces offer a peaceful environment perfect for reflection and relaxation. Enjoy the tranquility before heading to dinner.
    *   **Location:** Stanford University Campus.
    *   **Cost:** Free.
*   **6:00 PM onwards: Affordable Dinner**
    For dinner, consider exploring the diverse and affordable dining options in downtown Palo Alto or back in Sunnyvale. Many casual restaurants offer a range of cuisines to suit your taste and budget.
    *   **Suggestion:** Look for local taquerias, casual cafes, or international eateries in downtown Palo Alto or Sunnyvale.

Enjoy your relaxing and artsy day trip!

--------------------------------------------------



---
## Part 2: Supercharging Agents with Custom Tools 🛠️

So far, we've used the powerful built-in `GoogleSearch` tool. But the true power of agents comes from connecting them to your own logic and data sources.

This is where **custom tools** come in. Let's explore three patterns for giving your agent new skills, using real-world, practical examples.

### 2.1 The Simple `FunctionTool`: Calling a Real-Time Weather API

The most direct way to create a tool is by writing a Python function. This is perfect for synchronous tasks like fetching data from an API.

**Key Concept:** The function's **docstring** is critical. The ADK uses it as the tool's official description, which the LLM reads to understand its purpose, parameters, and when to use it.

In this example, we'll create a tool that calls the **free, public U.S. National Weather Service API** to get a real-time forecast. No API key needed!

In [13]:
# --- Tool Definition: A function that calls a live public API ---
import requests
import json

# A simple lookup to avoid needing a separate geocoding API for this example
LOCATION_COORDINATES = {
    "sunnyvale": "37.3688,-122.0363",
    "san francisco": "37.7749,-122.4194",
    "lake tahoe": "39.0968,-120.0324"
}

def get_live_weather_forecast(location: str) -> dict:
    """Gets the current, real-time weather forecast for a specified location in the US.

    Args:
        location: The city name, e.g., "San Francisco".

    Returns:
        A dictionary containing the temperature and a detailed forecast.
    """
    print(f"🛠️ TOOL CALLED: get_live_weather_forecast(location='{location}')")

    # Find coordinates for the location
    normalized_location = location.lower()
    coords_str = None
    for key, val in LOCATION_COORDINATES.items():
        if key in normalized_location:
            coords_str = val
            break
    if not coords_str:
        return {"status": "error", "message": f"I don't have coordinates for {location}."}

    try:
        # NWS API requires 2 steps: 1. Get the forecast URL from the coordinates.
        points_url = f"https://api.weather.gov/points/{coords_str}"
        headers = {"User-Agent": "ADK Example Notebook"}
        points_response = requests.get(points_url, headers=headers)
        points_response.raise_for_status() # Raise an exception for bad status codes
        forecast_url = points_response.json()['properties']['forecast']

        # 2. Get the actual forecast from the URL.
        forecast_response = requests.get(forecast_url, headers=headers)
        forecast_response.raise_for_status()

        # Extract the relevant forecast details
        current_period = forecast_response.json()['properties']['periods'][0]
        return {
            "status": "success",
            "temperature": f"{current_period['temperature']}°{current_period['temperatureUnit']}",
            "forecast": current_period['detailedForecast']
        }
    except requests.exceptions.RequestException as e:
        return {"status": "error", "message": f"API request failed: {e}"}

# --- Agent Definition: An agent that USES the new tool ---

weather_agent = Agent(
    name="weather_aware_planner",
    model="gemini-2.5-flash",
    description="A trip planner that checks the real-time weather before making suggestions.",
    instruction="You are a cautious trip planner. Before suggesting any outdoor activities, you MUST use the `get_live_weather_forecast` tool to check conditions. Incorporate the live weather details into your recommendation.",
    tools=[get_live_weather_forecast]
)

print(f"🌦️ Agent '{weather_agent.name}' is created and can now call a live weather API!")

🌦️ Agent 'weather_aware_planner' is created and can now call a live weather API!


In [14]:
# --- Let's test the Weather-Aware Planner ---

async def run_weather_planner_test():
    weather_session = await session_service.create_session(app_name=weather_agent.name, user_id=my_user_id)
    query = "I want to go hiking near Lake Tahoe, what's the weather like?"
    print(f"🗣️ User Query: '{query}'")
    await run_agent_query(weather_agent, query, weather_session, my_user_id)

await run_weather_planner_test()

🗣️ User Query: 'I want to go hiking near Lake Tahoe, what's the weather like?'

🚀 Running query for agent: 'weather_aware_planner' in session: '1f94ac99-c6ab-40a3-ab8f-0341e95983a4'...


EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'location': 'Lake Tahoe'
        },
        id='adk-c043897e-7b04-49c7-9508-aca6bd9ee207',
        name='get_live_weather_forecast'
      ),
      thought_signature=b'\n\xad\x02\x01\x8f=k_\x8a\xdd9I\xbcbr33>`.\x08~L\xca\xf9\x97pZ\xb6\x14w\xeaB0\xf8Y\xba\x9d\xad_\x03\xd0\x8d\xfd!\x06YO"b#K\xc8\xc4\xebU2\xec\xf5\x81UyL\x95\xeeO\'f\xc2\xc2Z\xdbB\xd6JC\t@1\xb7\xb8P4\x02\xc3\x97\xef\\\xc9\xb3\xfer\xc3\xb1\xd5\x0e\xac...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=None interrupted=None custom_metadata=None usage_metadata=GenerateContentResponseUsageMetadata(
  candidates_token_count=10,
  candidates_tokens_details=[
    ModalityTokenCount(
      modality=<MediaModality.TEXT: 'TEXT'>,
      token_count=10
    ),
  ],
  prompt_token_count=161,
  pr

The weather in Lake Tahoe is sunny with a high of 53°F. There will be an east wind blowing 10 to 20 mph, with gusts as high as 35 mph. It sounds like a good day for a hike, but be prepared for some strong winds!

--------------------------------------------------



## 2.2 The Agent-as-a-Tool: Consulting a Specialist 🧑‍🍳

Why build one agent that does everything when you can build a **team of specialist agents?** The **Agent-as-a-Tool** pattern allows one agent to delegate a task to another agent.

**Key Concept:** This is different from a sub-agent. When Agent A calls Agent B as a tool, Agent B's response is passed **back to Agent A**. Agent A then uses that information to form its own final response to the user. It's a powerful way to compose complex behaviors from simpler, focused, and reusable agents.

### How It Works

Our top-level agent, the `trip_data_concierge_agent`, acts as the **Orchestrator**. It has two tools at its disposal:

1.  `call_db_agent`: A function that internally calls our `db_agent` to fetch raw data.
2.  `call_concierge_agent`: A function that calls the `concierge_agent`.

The `concierge_agent` itself has a tool: the `food_critic_agent`.

The flow for a complex query is:

1.  **User** asks the `trip_data_concierge_agent` for a hotel and a nearby restaurant.
2.  The **Orchestrator** first calls `call_db_agent` to get hotel data.
3.  The data is saved in `tool_context.state`.
4.  The **Orchestrator** then calls `call_concierge_agent`, which retrieves the hotel data from the context.
5.  The `concierge_agent` receives the request and decides it needs to use its own tool, the `food_critic_agent`.
6.  The `food_critic_agent` provides a witty recommendation.
7.  The `concierge_agent` gets the critic's response and politely formats it.
8.  This final, polished response is returned to the **Orchestrator**, which presents it to the user.

                         +-----------------------------------------------------------+
                         |              🧭 Trip Data Concierge Agent                 |
                         |-----------------------------------------------------------|
                         |  Model: gemini-2.5-flash                                  |
                         |  Description:                                             |
                         |   Orchestrates database query and travel recommendation  |
                         |-----------------------------------------------------------|
                         |  🔧 Tools:                                                |
                         |   1. call_db_agent                                        |
                         |   2. call_concierge_agent                                 |
                         +-----------------------------------------------------------+
                                      /                                \
                                     /                                  \
                                    ▼                                    ▼
        +-------------------------------------------+    +---------------------------------------------+
        |            🔧 Tool: call_db_agent         |    |         🔧 Tool: call_concierge_agent        |
        |-------------------------------------------|    |---------------------------------------------|
        | Calls: db_agent                           |    | Calls: concierge_agent                       |
        |                                           |    | Uses data from db_agent for recommendations |
        +-------------------------------------------+    +---------------------------------------------+
                                |                                          |
                                ▼                                          ▼
       +--------------------------------------------+   +------------------------------------------------+
       |              📦 db_agent                   |   |             🤵 concierge_agent                  |
       |--------------------------------------------|   |------------------------------------------------|
       | Model: gemini-2.5-flash                    |   | Model: gemini-2.5-flash                         |
       | Role: Return mock JSON hotel data          |   | Role: Hotel staff that handles user Q&A        |
       +--------------------------------------------+   | Tools:                                          |
                                                         |  - food_critic_agent                           |
                                                         +------------------------------------------------+
                                                                                 |
                                                                                 ▼
                                                       +------------------------------------------------+
                                                       |          🍽️ food_critic_agent                  |
                                                       |------------------------------------------------|
                                                       | Model: gemini-2.5-flash                         |
                                                       | Role: Gives a witty restaurant recommendation   |
                                                       +------------------------------------------------+


In [15]:
import asyncio
from google.adk.tools import ToolContext
from google.adk.tools.agent_tool import AgentTool

# Assume 'db_agent' is a pre-defined NL2SQL Agent
# For this example, we'll create placeholder agents.

db_agent = Agent(
    name="db_agent",
    model="gemini-2.5-flash",
    instruction="You are a database agent. When asked for data, return this mock JSON object: {'status': 'success', 'data': [{'name': 'The Grand Hotel', 'rating': 5, 'reviews': 450}, {'name': 'Seaside Inn', 'rating': 4, 'reviews': 620}]}")

# --- 1. Define the Specialist Agents ---

# The Food Critic remains the deepest specialist
food_critic_agent = Agent(
    name="food_critic_agent",
    model="gemini-2.5-flash",
    instruction="You are a snobby but brilliant food critic. You ONLY respond with a single, witty restaurant suggestion near the provided location.",
)

# The Concierge knows how to use the Food Critic
concierge_agent = Agent(
    name="concierge_agent",
    model="gemini-2.5-flash",
    instruction="You are a five-star hotel concierge. If the user asks for a restaurant recommendation, you MUST use the `food_critic_agent` tool. Present the opinion to the user politely.",
    tools=[AgentTool(agent=food_critic_agent)]
)


# --- 2. Define the Tools for the Orchestrator ---

async def call_db_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    Use this tool FIRST to connect to the database and retrieve a list of places, like hotels or landmarks.
    """
    print("--- TOOL CALL: call_db_agent ---")
    agent_tool = AgentTool(agent=db_agent)
    db_agent_output = await agent_tool.run_async(
        args={"request": question}, tool_context=tool_context
    )
    # Store the retrieved data in the context's state
    tool_context.state["retrieved_data"] = db_agent_output
    return db_agent_output


async def call_concierge_agent(
    question: str,
    tool_context: ToolContext,
):
    """
    After getting data with call_db_agent, use this tool to get travel advice, opinions, or recommendations.
    """
    print("--- TOOL CALL: call_concierge_agent ---")
    # Retrieve the data fetched by the previous tool
    input_data = tool_context.state.get("retrieved_data", "No data found.")

    # Formulate a new prompt for the concierge, giving it the data context
    question_with_data = f"""
    Context: The database returned the following data: {input_data}

    User's Request: {question}
    """

    agent_tool = AgentTool(agent=concierge_agent)
    concierge_output = await agent_tool.run_async(
        args={"request": question_with_data}, tool_context=tool_context
    )
    return concierge_output


# --- 3. Define the Top-Level Orchestrator Agent ---

trip_data_concierge_agent = Agent(
    name="trip_data_concierge",
    model="gemini-2.5-flash",
    description="Top-level agent that queries a database for travel data, then calls a concierge agent for recommendations.",
    tools=[call_db_agent, call_concierge_agent],
    instruction="""
    You are a master travel planner who uses data to make recommendations.

    1.  **ALWAYS start with the `call_db_agent` tool** to fetch a list of places (like hotels) that match the user's criteria.

    2.  After you have the data, **use the `call_concierge_agent` tool** to answer any follow-up questions for recommendations, opinions, or advice related to the data you just found.
    """,
)

print(f"✅ Orchestrator Agent '{trip_data_concierge_agent.name}' is defined and ready.")

✅ Orchestrator Agent 'trip_data_concierge' is defined and ready.


In [16]:
# --- Let's test the Trip Data Concierge Agent ---

async def run_trip_data_concierge():
    """
    Sets up a session and runs a query against the top-level
    trip_data_concierge_agent.
    """
    # Create a new, single-use session for this query
    concierge_session = await session_service.create_session(
        app_name=trip_data_concierge_agent.name,
        user_id=my_user_id
    )

    # This query is specifically designed to trigger the full two-step process:
    # 1. Get data from the db_agent.
    # 2. Get a recommendation from the concierge_agent based on that data.
    query = "Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews."
    print(f"🗣️ User Query: '{query}'")

    # We call our existing helper function with the top-level orchestrator agent
    await run_agent_query(trip_data_concierge_agent, query, concierge_session, my_user_id)

# Run the test
await run_trip_data_concierge()

🗣️ User Query: 'Find the top-rated hotels in San Francisco from the database, then suggest a dinner spot near the one with the most reviews.'

🚀 Running query for agent: 'trip_data_concierge' in session: 'e2d86d9e-3618-41b9-b45e-e1276582eaa4'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      function_call=FunctionCall(
        args={
          'question': 'Find the top-rated hotels in San Francisco'
        },
        id='adk-70ab16ec-21cc-4c5b-8e97-277f9be1e8df',
        name='call_db_agent'
      ),
      thought_signature=b'\n\xe9\x04\x01\x8f=k_\xe1O\x8f\x9epsqA\xc4\xe8\n\xb8@\xdc\x02]\x9ah\xe5\xbbx\x00Q\xcfg\xeaJ-\r\xcf\xf9\x13I\xe7\xdd\xd1\xe94;\x9e\xec\xa0\x98\xdf1\xf3\x08\xaf\x89\xae\x93\xdc\xb3\xee\x1b9\r\x937A\xc5\xee\xf3\x9e\xd3uJ\xe2\x01DLB,\x02\xdaz|\xffc\x15@.\xe9U\xa5\xec9)\xac...'
    ),
  ],
  role='model'
) grounding_metadata=None partial=None turn_complete=None finish_reason=<FinishReason.STOP: 'STOP'> error_code=None error_message=

I recommend "The Saltwater Siren" near the Seaside Inn. The food critic describes it as a place for those whose "palate demands more than mere coastal clichés." I hope you enjoy your dinner!

--------------------------------------------------



---
## Part 3: Agent with a Memory - The Adaptive Planner 🗺️

Now, let's see an agent that not only **remembers** but also **adapts**. We'll challenge the `multi_day_trip_agent` to re-plan part of its itinerary based on our feedback. This is a much more realistic test of conversational AI.

```
+-----------------------------------------------------+
|         Adaptive Multi-Day Trip Agent 🗺️           |
|-----------------------------------------------------|
|  Model: gemini-2.5-flash                            |
|  Description:                                       |
|   Builds multi-day travel itineraries step-by-step, |
|   remembers previous days, adapts to feedback       |
|-----------------------------------------------------|
|  🔧 Tools:                                          |
|   - Google Search                                   |
|-----------------------------------------------------|
|  🧠 Capabilities:                                   |
|   - Memory of past conversation & preferences       |
|   - Progressive planning (1 day at a time)          |
|   - Adapts to user feedback                         |
|   - Ensures activity variety across days            |
+-----------------------------------------------------+

            ▲
            |
    +---------------------------+
    |     User Interaction      |
    |---------------------------|
    | - Destination             |
    | - Trip duration           |
    | - Interests & feedback    |
    +---------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Day-by-Day Itinerary Generation              |
|-----------------------------------------------------|
|  🗓️ Day N Output (Markdown format):                 |
|   - Morning / Afternoon / Evening activities        |
|   - Personalized & context-aware                    |
|   - Changes accepted, feedback acknowledged         |
+-----------------------------------------------------+

            |
            ▼

+-----------------------------------------------------+
|        Next Day Planning Triggered 🚀               |
|-----------------------------------------------------|
| - Builds on prior days                              |
| - Avoids repetition                                 |
| - Asks user for confirmation before proceeding      |
+-----------------------------------------------------+
```


In [17]:
# --- Agent Definition: The Adaptive Planner ---

def create_multi_day_trip_agent():
    """Create the Progressive Multi-Day Trip Planner agent"""
    return Agent(
        name="multi_day_trip_agent",
        model="gemini-2.5-flash",
        description="Agent that progressively plans a multi-day trip, remembering previous days and adapting to user feedback.",
        instruction="""
        You are the "Adaptive Trip Planner" 🗺️ - an AI assistant that builds multi-day travel itineraries step-by-step.

        Your Defining Feature:
        You have short-term memory. You MUST refer back to our conversation to understand the trip's context, what has already been planned, and the user's preferences. If the user asks for a change, you must adapt the plan while keeping the unchanged parts consistent.

        Your Mission:
        1.  **Initiate**: Start by asking for the destination, trip duration, and interests.
        2.  **Plan Progressively**: Plan ONLY ONE DAY at a time. After presenting a plan, ask for confirmation.
        3.  **Handle Feedback**: If a user dislikes a suggestion (e.g., "I don't like museums"), acknowledge their feedback, and provide a *new, alternative* suggestion for that time slot that still fits the overall theme.
        4.  **Maintain Context**: For each new day, ensure the activities are unique and build logically on the previous days. Do not suggest the same things repeatedly.
        5.  **Final Output**: Return each day's itinerary in MARKDOWN format.
        """,
        tools=[google_search]
    )

multi_day_agent = create_multi_day_trip_agent()
print(f"🗺️ Agent '{multi_day_agent.name}' is created and ready to plan and adapt!")

🗺️ Agent 'multi_day_trip_agent' is created and ready to plan and adapt!


### Scenario 3a: Agent WITH Memory (Using a SINGLE Session) ✅

First, let's see the correct way to do it. We will use the **exact same `trip_session` object** for the entire conversation. Watch how the agent remembers the context from Turn 1 to correctly handle the requests in Turn 2 and 3.

In [18]:
# --- Scenario 2: Testing Adaptation and Memory ---

async def run_adaptive_memory_demonstration():
    print("### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###")

    # Create ONE session that we will reuse for the whole conversation
    trip_session = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"Created a single session for our trip: {trip_session.id}")

    # --- Turn 1: The user initiates the trip ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    print(f"\n🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, trip_session, my_user_id)

    # --- Turn 2: The user gives FEEDBACK and asks for a CHANGE ---
    # We use the EXACT SAME `trip_session` object!
    query2 = "That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?"
    print(f"\n🗣️ User (Turn 2 - Feedback): '{query2}'")
    await run_agent_query(multi_day_agent, query2, trip_session, my_user_id)

    # --- Turn 3: The user confirms and asks to continue ---
    query3 = "Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind."
    print(f"\n🗣️ User (Turn 3 - Confirmation): '{query3}'")
    await run_agent_query(multi_day_agent, query3, trip_session, my_user_id)

await run_adaptive_memory_demonstration()

### 🧠 DEMO 2: AGENT THAT ADAPTS (SAME SESSION) ###
Created a single session for our trip: 28e07889-f1c9-4f15-ba68-d427cfeb3b6a

🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '28e07889-f1c9-4f15-ba68-d427cfeb3b6a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Hello! A 2-day trip to Lisbon sounds wonderful. I can certainly help you plan that, focusing on historic sites and delicious local food.

Let's start with **Day 1**. How about this for your first day in Lisbon?

**Day 1: Discovering Belém's History and Flavors**

*   **Morning (9:00 AM - 1:00 PM): Explore Belém's UNESCO World Heritage Sites.**
    *   Begin at the iconic **Belém Tower (Torre de Belém)**, a 16th-century fortification that played a significant role in Portugal's Age of Discoveries.
    *   Just a short walk away, visit the magn

Hello! A 2-day trip to Lisbon sounds wonderful. I can certainly help you plan that, focusing on historic sites and delicious local food.

Let's start with **Day 1**. How about this for your first day in Lisbon?

**Day 1: Discovering Belém's History and Flavors**

*   **Morning (9:00 AM - 1:00 PM): Explore Belém's UNESCO World Heritage Sites.**
    *   Begin at the iconic **Belém Tower (Torre de Belém)**, a 16th-century fortification that played a significant role in Portugal's Age of Discoveries.
    *   Just a short walk away, visit the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, another stunning example of Manueline architecture and a resting place for Vasco da Gama.
*   **Lunch (1:00 PM - 2:30 PM): Indulge in local delights.**
    *   Head to the famous **Pastéis de Belém** for freshly baked custard tarts – a true Lisbon specialty.
    *   For a savory option, explore the local eateries around Belém for traditional Portuguese dishes like Bacalhau à Brás (codfish with scrambled eggs and potatoes) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Maritime History and Riverside Stroll.**
    *   Visit the **Monument to the Discoveries (Padrão dos Descobrimentos)**, celebrating Portugal's explorers.
    *   Enjoy a leisurely stroll along the Tagus River, taking in the views and the historic atmosphere of the Belém district.
*   **Evening (7:30 PM onwards): Authentic Dinner in Alcântara.**
    *   Take a short taxi or tram ride to the nearby Alcântara district, known for its vibrant atmosphere and excellent restaurants.
    *   Enjoy a traditional Portuguese dinner, perhaps trying a "cataplana" (seafood stew) or some grilled fish.

How does this sound for your first day? We can adjust anything you like!

--------------------------------------------------


🗣️ User (Turn 2 - Feedback): 'That sounds pretty good, but I'm not a huge fan of castles. Can you replace the morning activity for Day 1 with something else historical?'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '28e07889-f1c9-4f15-ba68-d427cfeb3b6a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Apologies! My mistake for including something you're not keen on. You're right, Belém Tower has a defensive, castle-like feel.

Let's adjust Day 1 to better suit your preferences while keeping you in the historic Belém area for a coherent day.

Here’s a revised plan for your first day:

**Day 1: Discovering Belém's Rich History and Flavors (Revised)**

*   **Morning (9:00 AM - 1:00 PM): Explore Belém's Historic Gems.**
    *   Begin at the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a stunning masterpiece of Manueline archit

Apologies! My mistake for including something you're not keen on. You're right, Belém Tower has a defensive, castle-like feel.

Let's adjust Day 1 to better suit your preferences while keeping you in the historic Belém area for a coherent day.

Here’s a revised plan for your first day:

**Day 1: Discovering Belém's Rich History and Flavors (Revised)**

*   **Morning (9:00 AM - 1:00 PM): Explore Belém's Historic Gems.**
    *   Begin at the magnificent **Jerónimos Monastery (Mosteiro dos Jerónimos)**, a UNESCO World Heritage site and a stunning masterpiece of Manueline architecture. You can explore its ornate church and tranquil cloisters.
    *   Afterward, visit the fascinating **National Coach Museum (Museu Nacional dos Coches)**, which houses one of the finest collections of historical carriages and coaches in the world. It offers a unique glimpse into royal transport and Portuguese history.
*   **Lunch (1:00 PM - 2:30 PM): Indulge in local delights.**
    *   Head to the famous **Pastéis de Belém** for freshly baked custard tarts – a true Lisbon specialty.
    *   For a savory option, explore the local eateries around Belém for traditional Portuguese dishes like Bacalhau à Brás (codfish with scrambled eggs and potatoes) or grilled sardines.
*   **Afternoon (2:30 PM - 6:00 PM): Maritime History and Riverside Stroll.**
    *   Visit the **Monument to the Discoveries (Padrão dos Descobrimentos)**, celebrating Portugal's explorers and their maritime achievements.
    *   Enjoy a leisurely stroll along the Tagus River, taking in the views and the historic atmosphere of the Belém district.
*   **Evening (7:30 PM onwards): Authentic Dinner in Alcântara.**
    *   Take a short taxi or tram ride to the nearby Alcântara district, known for its vibrant atmosphere and excellent restaurants.
    *   Enjoy a traditional Portuguese dinner, perhaps trying a "cataplana" (seafood stew) or some grilled fish.

How does this revised Day 1 sound to you? We've replaced the fortification with another unique historical museum!

--------------------------------------------------


🗣️ User (Turn 3 - Confirmation): 'Yes, the new plan for Day 1 is perfect! Please plan Day 2 now, keeping the food theme in mind.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '28e07889-f1c9-4f15-ba68-d427cfeb3b6a'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Excellent! I'm glad Day 1 is to your liking. Let's move on to planning your second day in beautiful Lisbon, ensuring we continue to explore historic sites and savor more delicious local food.

Here's a proposal for **Day 2**:

**Day 2: Exploring Alfama, Baixa, and Lisbon's Culinary Delights**

*   **Morning (9:30 AM - 1:00 PM): Wander through Historic Alfama.**
    *   Start your day at the majestic **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, which showcases a blend of Romanesque, Gothic, and Baroque architectural styles. It's a true historical gem.
    *   Next, immerse yourself in **Alfama**, 

Excellent! I'm glad Day 1 is to your liking. Let's move on to planning your second day in beautiful Lisbon, ensuring we continue to explore historic sites and savor more delicious local food.

Here's a proposal for **Day 2**:

**Day 2: Exploring Alfama, Baixa, and Lisbon's Culinary Delights**

*   **Morning (9:30 AM - 1:00 PM): Wander through Historic Alfama.**
    *   Start your day at the majestic **Lisbon Cathedral (Sé de Lisboa)**, the city's oldest church, which showcases a blend of Romanesque, Gothic, and Baroque architectural styles. It's a true historical gem.
    *   Next, immerse yourself in **Alfama**, Lisbon's oldest and most atmospheric district. Lose yourself in its labyrinthine alleys, winding staircases, and charming squares. Discover hidden viewpoints (miradouros) like **Miradouro das Portas do Sol** or **Miradouro de Santa Luzia**, offering breathtaking panoramic views of the city's rooftops, the Tagus River, and the iconic São Jorge Castle from afar.
    *   Consider a brief visit to the **Fado Museum** to learn about the history of Portugal's soulful national music genre.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Flavors in Alfama.**
    *   Dine at a charming, traditional "tasca" (a small, informal restaurant) tucked away in Alfama. This is the perfect opportunity to savor authentic Portuguese home-style cooking.
    *   After lunch, don't miss trying a shot of **Ginjinha**, the traditional Portuguese sour cherry liqueur, often served in a small chocolate cup, from one of the tiny local vendors.
*   **Afternoon (2:30 PM - 6:00 PM): Baixa, Chiado, and an Iconic Ride.**
    *   Walk through **Baixa**, the elegant downtown area meticulously rebuilt after the 1755 earthquake, characterized by its neoclassical architecture and impressive grid-patterned streets. Explore bustling **Rossio Square** and the grand **Praça do Comércio (Commerce Square)** right on the riverfront.
    *   Experience a ride on the iconic **Tram 28** (if time and crowds permit) for a scenic journey through some of Lisbon's historic neighborhoods.
    *   Explore **Chiado**, an elegant and bohemian district known for its historic cafes, beautiful theaters, and upscale shops.
*   **Evening (7:30 PM onwards): Fado and Gastronomy.**
    *   Conclude your trip with an authentic **Fado show** paired with dinner. Many traditional Fado houses in Alfama or the lively Bairro Alto district offer this quintessential Lisbon experience, recognized by UNESCO as a World Heritage item. It's a wonderful way to connect with the soul of Portuguese culture and enjoy another fantastic meal.

How does this plan for your second day in Lisbon sound?

--------------------------------------------------



### Scenario 3b: Agent WITHOUT Memory (Using SEPARATE Sessions) ❌

Now, let's see what happens if we mess up our session management. Here, we'll give the agent a case of amnesia by creating a **brand new, separate session for each turn**.

Pay close attention to the agent's response to the second query. Because it's in a new session, it has no memory of the trip to Lisbon we just discussed!

In [19]:
# --- Scenario 2b: Demonstrating Memory FAILURE ---

async def run_memory_failure_demonstration():
    print("\n" + "#"*60)
    print("### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###")
    print("#"*60)

    # --- Turn 1: The user initiates the trip in the FIRST session ---
    query1 = "Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food."
    session_one = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a session for Turn 1: {session_one.id}")
    print(f"🗣️ User (Turn 1): '{query1}'")
    await run_agent_query(multi_day_agent, query1, session_one, my_user_id)

    # --- Turn 2: The user asks to continue... but in a completely NEW session ---
    query2 = "Yes, that looks perfect! Please plan Day 2."
    session_two = await session_service.create_session(
        app_name=multi_day_agent.name,
        user_id=my_user_id
    )
    print(f"\nCreated a BRAND NEW session for Turn 2: {session_two.id}")
    print(f"🗣️ User (Turn 2): '{query2}'")
    await run_agent_query(multi_day_agent, query2, session_two, my_user_id)

await run_memory_failure_demonstration()


############################################################
### 🧠 DEMO 2b: AGENT WITH AMNESIA (SEPARATE SESSIONS) ###
############################################################

Created a session for Turn 1: 3a5ddd63-3fde-4a4a-996d-8f31063c4345
🗣️ User (Turn 1): 'Hi! I want to plan a 2-day trip to Lisbon, Portugal. I'm interested in historic sites and great local food.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '3a5ddd63-3fde-4a4a-996d-8f31063c4345'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Great! Lisbon is a fantastic choice for historic sites and delicious local food. I'm excited to help you plan your 2-day adventure.

Let's start with **Day 1**. How about we explore the historic Alfama district and São Jorge Castle, followed by some traditional Portuguese cuisine?

Here’s a possible itinerary for your first day:

### Day 1: Historic Alfama & Castle Views

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & 

Great! Lisbon is a fantastic choice for historic sites and delicious local food. I'm excited to help you plan your 2-day adventure.

Let's start with **Day 1**. How about we explore the historic Alfama district and São Jorge Castle, followed by some traditional Portuguese cuisine?

Here’s a possible itinerary for your first day:

### Day 1: Historic Alfama & Castle Views

*   **Morning (9:00 AM - 1:00 PM): Explore Alfama & São Jorge Castle**
    *   Begin your day by wandering through the narrow, winding streets of **Alfama**, Lisbon's oldest district. Discover hidden viewpoints (miradouros), traditional Fado houses, and charming squares.
    *   Head up to **São Jorge Castle (Castelo de São Jorge)**, a majestic Moorish castle offering panoramic views of the city and the Tagus River. Explore its ancient walls, archaeological site, and peacocks roaming freely.
*   **Lunch (1:00 PM - 2:30 PM): Traditional Portuguese Lunch in Alfama**
    *   Enjoy a traditional Portuguese meal in a local tasca (tavern) within Alfama. Look for dishes like *Bacalhau à Brás* (codfish with scrambled eggs and potatoes) or *Sardinhas Assadas* (grilled sardines, especially in warmer months).
*   **Afternoon (2:30 PM - 6:00 PM): Miradouros & Tram 28 Ride**
    *   After lunch, explore more viewpoints like **Miradouro das Portas do Sol** and **Miradouro de Santa Luzia** for stunning photo opportunities.
    *   Catch the iconic **Tram 28** for a scenic ride through historic neighborhoods, including Alfama, Graça, and Baixa. It's a great way to see many of Lisbon's sights.
*   **Evening (7:30 PM onwards): Dinner with Fado Music**
    *   Experience authentic Portuguese culture with dinner at a restaurant in Alfama or Bairro Alto that offers live **Fado music**. Savor more local dishes while listening to this soulful traditional music.

How does this sound for your first day in Lisbon? We can adjust anything you like!

--------------------------------------------------


Created a BRAND NEW session for Turn 2: 39312485-41ff-4022-8147-d87c3ffea73c
🗣️ User (Turn 2): 'Yes, that looks perfect! Please plan Day 2.'

🚀 Running query for agent: 'multi_day_trip_agent' in session: '39312485-41ff-4022-8147-d87c3ffea73c'...
EVENT: model_version='gemini-2.5-flash' content=Content(
  parts=[
    Part(
      text="""Okay, great! Let's plan out **Day 2** in Paris. I'll focus on a different area of the city to give you a varied experience, keeping your interests in art, history, local cuisine, and hidden gems in mind.

Here's a possible plan for Day 2:

### **Day 2: Bohemian Charm and Artistic Flavors**

*   **Morning (9:00 AM - 1:00 PM): Explore Le Marais**
    *   Begin your day by wandering through the historic and vibrant **Le Marais district**. This area is packed with beautiful 17th-century architecture, charming cobblestone streets, hidden courtyards, and trendy boutiques. It's a great place to discover local 

Okay, great! Let's plan out **Day 2** in Paris. I'll focus on a different area of the city to give you a varied experience, keeping your interests in art, history, local cuisine, and hidden gems in mind.

Here's a possible plan for Day 2:

### **Day 2: Bohemian Charm and Artistic Flavors**

*   **Morning (9:00 AM - 1:00 PM): Explore Le Marais**
    *   Begin your day by wandering through the historic and vibrant **Le Marais district**. This area is packed with beautiful 17th-century architecture, charming cobblestone streets, hidden courtyards, and trendy boutiques. It's a great place to discover local history and some lesser-known spots.
    *   Visit the **Place des Vosges**, Paris's oldest planned square, known for its elegant red-brick facades and arcades.
    *   Explore the **Rue des Rosiers**, the heart of the Jewish Quarter, famous for its falafel stands and unique shops.
*   **Lunch (1:00 PM - 2:00 PM): Local Delights in Le Marais**
    *   Grab a delicious falafel from one of the renowned spots on Rue des Rosiers (like L'As du Fallafel) or choose a cozy café for a traditional French crêpe or quiche.
*   **Afternoon (2:00 PM - 6:00 PM): Modern Art at Centre Pompidou or Classic Art at Musée d'Orsay**
    *   **Option 1 (Modern Art & Architecture):** Head to the **Centre Pompidou**. This iconic high-tech building houses the National Museum of Modern Art, featuring an impressive collection from the 20th and 21st centuries. Don't miss the panoramic views of Paris from its top floor.
    *   **Option 2 (Impressionist Art):** If modern art isn't quite your preference or you enjoyed a lot of classical art on Day 1, consider the **Musée d'Orsay**. Housed in a stunning former railway station, it boasts an unparalleled collection of Impressionist and Post-Impressionist masterpieces (Monet, Renoir, Van Gogh, etc.).
*   **Evening (7:00 PM onwards): Dinner and Jazz in the Latin Quarter**
    *   Cross the Seine to the **Latin Quarter**, known for its bohemian atmosphere, universities, and lively nightlife.
    *   Enjoy dinner at a traditional French bistro in this historic district.
    *   After dinner, seek out a jazz club for some live music, a classic Parisian evening experience.

How does this sound for your second day? We can adjust anything you like!

--------------------------------------------------



See? The agent was confused! It likely asked what destination or what trip we were talking about. Because the second query was in a fresh, isolated session, the agent had no memory of planning Day 1 in Lisbon.

This perfectly illustrates why **managing sessions is the key to building truly conversational agents!**

---
## 🎉 Congratulations! 🎉

Congratulations on completing your ADK adventure into Tools and Memory! You've taken a massive leap from building single-shot agents to creating dynamic, stateful AI systems.

Let's recap the powerful concepts you've mastered:

- **Fundamental Agent & Tools**: You started by building a "Day Trip Genie" and equipped it with its first tool, GoogleSearch.

- **Custom Function Tools**: You gave your agent a new sense by creating a custom tool to fetch live data from the U.S. National Weather Service API.

- **Agent-as-a-Tool**: You orchestrated a sophisticated hierarchy where agents delegate tasks to other, more specialized agents, creating a collaborative team.

- **The Power of Memory**: Most importantly, you saw firsthand how managing a single, persistent Session allows an agent to remember context, adapt to user feedback, and conduct a meaningful, multi-turn conversation.

```
   __            /\_/\         /\_/\        /\_/\         __             (\__/)
o-''|\_____/).  ( o.o )       ( -.- )      ( ^_^ )     o-''|\_____/).    ( ^_^ )
 \_/|_)     )    > ^ <         > * <        >💖<         \_/|_)     )     / >🌸< \
    \  __  /                                              \  __  /         /   \
    (_/ (_/                                               (_/ (_/        (___|___)
```
